# Smart Money Concepts with qust

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


本节 展示如何在 qust 里使用 `smc` 命名空间计算 Smart Money Concepts 指标，并用 qust monitor 画图。这里的 `smc` 思路参考了开源项目 [joshyattridge/smart-money-concepts](https://github.com/joshyattridge/smart-money-concepts)，指标列表和字段含义参考它的 [README indicators 章节](https://github.com/joshyattridge/smart-money-concepts#indicators)。

`smart-money-concepts` 原项目把 SMC 描述为一组用于观察市场情绪、趋势结构和潜在反转的指标工具。它的概念来源包括 ICT / Inner Circle Trader 常用的 Order Block、Liquidity、Fair Value Gap、Swing High/Low、Break of Structure、Change of Character 等。换成更工程化的说法：这些指标不是单个“买卖点公式”，而是一组把 K 线结构转成可计算事件的工具。

在 qust 里，SMC 指标不是独立函数，而是表达式链的一部分：

```python
col("open", "high", "low", "close").smc.fvg()
```

这带来几个好处：

1. 可以和其它 qust 算子组合，例如 `.filter(...)`、`.with_cols(...)`、`.over("ticker", "ct")`、`.monitor...plot(...)`；
2. 可以把 SMC 输出直接接入策略信号、因子分析、回测或监控；
3. 多品种数据里可以通过 `.over("ticker", "ct")` 独立维护结构；
4. 输入和输出都是 DataFrame 列，方便检查、排序、筛选和画图。

重要提醒：SMC 指标适合做结构观察、信号过滤和研究特征，不应该单独作为交易决策。原项目 README 也明确强调它是教育用途，交易需要风险管理和独立验证。


In [1]:
import qust as qs

import qust.future.future  # 注册 kline / monitor / strategy 等领域 namespace
import qust.smc            # 注册 smc namespace
from qust import col
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(20)


polars.config.Config

## 1. 数据准备

SMC 指标通常输入 OHLC 或 OHLCV。原项目要求 DataFrame 至少包含小写列名 `open/high/low/close`，部分指标还需要 `volume`。qust 这里更强调表达式输入顺序：只要你写的是

```python
col("open", "high", "low", "close")
```

算子就按这四列顺序读取，不强制列名必须小写。但是顺序必须正确，否则指标语义会错。

示例从 GitHub 读取 K 线数据，并取 `au` 的前 1200 行。样例数据字段：

- `ticker`: 品种；
- `datetime`: K 线时间；
- `open/high/low/close`: OHLC；
- `volume`: 成交量；
- `is_finished`: K 线是否完成。


In [2]:
DATA_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"

raw_kline = pl.read_parquet(DATA_PATH)
smc_data = (
    raw_kline
    .filter(pl.col("ticker") == "au")
    .head(1200)
)

print("raw shape:", raw_kline.shape)
print("smc sample shape:", smc_data.shape)
smc_data.head(5)


raw shape: (408782, 8)
smc sample shape: (0, 8)


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64


## 2. 原库 API 与 qust API 对照

`smart-money-concepts` 原库一般是函数式调用，例如 `smc.fvg(ohlc)`。qust 里是表达式调用，例如：

```python
col("open", "high", "low", "close").smc.fvg(join_consecutive=True)
```

二者的核心差别是：原库每次调用一个函数并返回一个 DataFrame；qust 里 SMC 是表达式节点，可以嵌入更长的数据流。

| 指标 | 原库 README 调用 | qust 表达式调用 | 主要输出 |
| --- | --- | --- | --- |
| Fair Value Gap | `smc.fvg(ohlc)` | `col("open", "high", "low", "close").smc.fvg()` | `FVG, Top, Bottom, MitigatedIndex` |
| Swing Highs/Lows | `smc.swing_highs_lows(ohlc)` | `col("open", "high", "low", "close").smc.swing_highs_lows()` | `HighLow, Level` |
| BOS/CHOCH | `smc.bos_choch(ohlc, swing_highs_lows)` | `col("open", "high", "low", "close").smc.bos_choch()` | `BOS, CHOCH, Level, BrokenIndex` |
| Order Blocks | `smc.ob(ohlc, swing_highs_lows)` | `col("open", "high", "low", "close", "volume").smc.ob()` | `OB, Top, Bottom, OBVolume, Percentage` |
| Liquidity | `smc.liquidity(ohlc, swing_highs_lows)` | `col("open", "high", "low", "close").smc.liquidity()` | `Liquidity, Level, End, Swept` |
| Previous High/Low | `smc.previous_high_low(ohlc)` | `col("datetime", "high", "low", "close").smc.previous_high_low()` | `PreviousHigh, PreviousLow, BrokenHigh, BrokenLow` |
| Sessions | `smc.sessions(ohlc)` | `col("datetime", "high", "low").smc.sessions(...)` | `Active, High, Low` |
| Retracements | `smc.retracements(ohlc, swing_highs_lows)` | `col("open", "high", "low", "close").smc.retracements()` | `Direction, CurrentRetracement%, DeepestRetracement%` |

### 2.1 Fair Value Gap, FVG

FVG 用来识别价格快速穿越后留下的不平衡区间。原 README 的定义可以理解为：

- 看当前 K 线是否是阳线；如果上一根 K 线的 high 低于下一根 K 线的 low，中间留下向上的空档，就是 bullish FVG；
- 看当前 K 线是否是阴线；如果上一根 K 线的 low 高于下一根 K 线的 high，中间留下向下的空档，就是 bearish FVG。

参数：

- `join_consecutive`: 如果连续出现多个同方向 FVG，可以合并成一个区间；qust 会用最高的 top 和最低的 bottom 做合并边界。

输出：

- `FVG`: `1` 表示 bullish FVG，`-1` 表示 bearish FVG，`0` 表示没有；
- `Top`: FVG 区间上沿；
- `Bottom`: FVG 区间下沿；
- `MitigatedIndex`: 后续第几根 K 线回补了这个缺口。

### 2.2 Swing Highs and Lows

Swing High/Low 用来找局部极值。原 README 的逻辑是：当前 high 如果是前后 `swing_length` 根 K 线范围里的最高 high，就是 swing high；当前 low 如果是前后 `swing_length` 根 K 线范围里的最低 low，就是 swing low。

参数：

- `swing_length`: 往前和往后各看多少根 K 线。它越大，识别出来的结构越慢、越粗；越小，事件越多、噪声也更大。

输出：

- `HighLow`: `1` 表示 swing high，`-1` 表示 swing low；
- `Level`: 对应的价格水平。

### 2.3 BOS 与 CHOCH

BOS 是 Break of Structure，表示价格突破了已有结构；CHOCH 是 Change of Character，表示结构性质可能发生变化。原库需要把 `swing_highs_lows` 的结果传给 `bos_choch`；qust 当前接口里内部会用同样的 swing 逻辑重新识别结构，调用上更短。

参数：

- `swing_length`: 用于识别 swing 的窗口；
- `close_break`: 如果为 `True`，用 close 是否突破结构位判断；如果为 `False`，用 high/low 是否突破判断。

输出：

- `BOS`: `1` bullish BOS，`-1` bearish BOS；
- `CHOCH`: `1` bullish CHOCH，`-1` bearish CHOCH；
- `Level`: 被突破的结构水平；
- `BrokenIndex`: 被突破结构来自哪一行。

### 2.4 Order Blocks, OB

Order Block 试图识别价格区间内曾经集中出现大量市场订单的位置。原 README 把它描述为“高数量市场订单存在的价格范围”。实盘研究里通常会把 OB 当作潜在支撑、压力、回踩或反弹区域。

参数：

- `swing_length`: 用于定位结构突破前的候选 block；
- `close_break`: qust 中对应是否按 close 判断结构突破。

输出：

- `OB`: `1` bullish order block，`-1` bearish order block；
- `Top`: order block 上沿；
- `Bottom`: order block 下沿；
- `OBVolume`: 当前和前两根附近成交量的组合估计；
- `Percentage`: 用于粗略衡量 block 强度，值来自强弱量的比例关系。

### 2.5 Liquidity

Liquidity 用来识别一组接近的高点或低点。原 README 的定义是：多个 high 在很小范围内聚集，或者多个 low 在很小范围内聚集，就形成流动性区域。交易研究里常把这些区域理解为止损、突破单或挂单可能集中的位置。

参数：

- `swing_length`: 先用 swing 逻辑找候选高低点；
- `range_percent`: 判断“很小范围”的比例阈值。

输出：

- `Liquidity`: `1` 表示上方 liquidity，`-1` 表示下方 liquidity；
- `Level`: liquidity 聚集价格；
- `End`: 最后一个 liquidity 水平所在行；
- `Swept`: 后续哪一行扫过这个 liquidity 区域。

### 2.6 Previous High and Low

Previous High/Low 返回上一个周期的 high 和 low。常见用途是观察价格是否突破上一日、上一周或上一月的高低点。

参数：

- `time_frame`: 支持 `15m`、`1H`、`4H`、`1D`、`1W`、`1M`。

输出：

- `PreviousHigh`: 上一周期高点；
- `PreviousLow`: 上一周期低点；
- `BrokenHigh`: 当前周期内是否已经突破上一周期高点；
- `BrokenLow`: 当前周期内是否已经跌破上一周期低点。

### 2.7 Sessions

Sessions 用来标记一根 K 线是否处于指定交易时段，并输出该 session 内截至当前的高低点。原 README 支持 Sydney、Tokyo、London、New York、Asian kill zone、London open kill zone、New York kill zone、London close kill zone 和 Custom。

参数：

- `session`: 预设 session 名称，或 `Custom`；
- `start_time/end_time`: 自定义 session 的开始和结束时间，格式如 `"09:00"`；
- `time_zone`: qust 当前保留为接口参数，naive datetime 按原始时间解释。

输出：

- `Active`: 当前 K 线是否在 session 内；
- `High`: session 内截至当前最高点；
- `Low`: session 内截至当前最低点。

### 2.8 Retracements

Retracements 计算从 swing high 或 swing low 开始的回撤百分比。它不是直接判断多空，而是把当前价格相对最近结构段的位置量化。

输出：

- `Direction`: `1` 表示 bullish retracement，`-1` 表示 bearish retracement；
- `CurrentRetracement%`: 当前回撤百分比；
- `DeepestRetracement%`: 当前结构段里出现过的最深回撤百分比。

### 2.9 qust 版本和原库版本的差异

本节重点展示 qust 表达式调用。它和原库不是逐行完全相同的实现包装，主要差异：

- 原库按函数接收 `ohlc` DataFrame；qust 按表达式列顺序读取输入；
- 原库的部分函数显式接收 `swing_highs_lows` 结果；qust 当前接口内部按 `swing_length` 重新计算；
- qust 输出可以直接进入 `monitor`、`group_by`、`over`、策略或回测链；
- 多品种时建议在外层接 `.over("ticker", "ct")`，保证每个品种独立识别结构。


In [3]:
smc_expr = col(
    "ticker",
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "volume",
    col("open", "high", "low", "close").smc.fvg(join_consecutive=True).add_suffix("_fvg"),
    col("open", "high", "low", "close").smc.swing_highs_lows(swing_length=12).add_suffix("_swing"),
    col("open", "high", "low", "close").smc.bos_choch(swing_length=12, close_break=True).add_suffix("_structure"),
    col("open", "high", "low", "close", "volume").smc.ob(swing_length=12).add_suffix("_ob"),
    col("open", "high", "low", "close").smc.liquidity(swing_length=12, range_percent=0.02).add_suffix("_liq"),
    col("datetime", "high", "low", "close").smc.previous_high_low(time_frame="1D").add_suffix("_prev"),
    col("datetime", "high", "low").smc.sessions("Custom", start_time="09:00", end_time="15:00").add_suffix("_session"),
    col("open", "high", "low", "close").smc.retracements(swing_length=12).add_suffix("_retr"),
)

smc_features = smc_expr.calc_data(smc_data)
print("features shape:", smc_features.shape)
smc_features.select(
    "datetime",
    "close",
    "FVG__fvg",
    "HighLow__swing",
    "BOS__structure",
    "CHOCH__structure",
    "OB__ob",
    "Liquidity__liq",
    "PreviousHigh__prev",
    "PreviousLow__prev",
    "Active__session",
).head(12)


features shape: (0, 36)


datetime,close,FVG__fvg,HighLow__swing,BOS__structure,CHOCH__structure,OB__ob,Liquidity__liq,PreviousHigh__prev,PreviousLow__prev,Active__session
datetime[ms],f64,i32,i32,i32,i32,i32,i32,f64,f64,i32


## 3. 事件数量总览

这一步把多个 SMC 指标转成事件计数。它不是交易信号，只是帮助你快速理解这段样例数据里哪些结构出现得多。

这个表可以回答几个基础问题：

- FVG 是否很多，如果很多，说明行情里存在较多局部不平衡区间；
- Swing 数量是否合理，如果过多，说明 `swing_length` 可能太小；
- BOS/CHOCH 是否稀少，如果太少，说明结构过滤过粗；
- Previous Break 很多时，说明价格经常突破上一周期高低点；
- Session Active 可以检查自定义 session 时间范围是否符合预期。

图形使用 qust monitor bar，第一列是事件名称，第二列是事件数量。


In [4]:
smc_event_counts = pl.DataFrame({
    "event": [
        "FVG",
        "Swing",
        "BOS",
        "CHOCH",
        "Order Block",
        "Liquidity",
        "Previous Break",
        "Session Active",
    ],
    "count": [
        smc_features.select((pl.col("FVG__fvg") != 0).sum()).item(),
        smc_features.select((pl.col("HighLow__swing") != 0).sum()).item(),
        smc_features.select((pl.col("BOS__structure") != 0).sum()).item(),
        smc_features.select((pl.col("CHOCH__structure") != 0).sum()).item(),
        smc_features.select((pl.col("OB__ob") != 0).sum()).item(),
        smc_features.select((pl.col("Liquidity__liq") != 0).sum()).item(),
        smc_features.select(((pl.col("BrokenHigh__prev") != 0) | (pl.col("BrokenLow__prev") != 0)).sum()).item(),
        smc_features.select((pl.col("Active__session") != 0).sum()).item(),
    ],
})

smc_event_counts


event,count
str,i64
"""FVG""",0
"""Swing""",0
"""BOS""",0
"""CHOCH""",0
"""Order Block""",0
"""Liquidity""",0
"""Previous Break""",0
"""Session Active""",0


In [23]:
smc_event_bar = col("event", "count").monitor("smc_event_counts", show_axis_label=True).bar().runtime()
smc_event_bar.plot(smc_event_counts, open_in_jupyter=True, auto_open=False, height=520)


## 4. K 线图查看价格结构

SMC 指标最好不要只看表。表能告诉你“哪里发生了事件”，K 线图能告诉你事件出现在什么价格结构里。

这里用 qust monitor 的 K 线图查看同一段价格。一个常见研究流程是：

1. 先用事件表找出 FVG、BOS、CHOCH、Liquidity 等事件行；
2. 再回到 K 线图观察这些行附近的价格结构；
3. 如果后续要写策略，再把这些事件变成过滤条件，而不是直接裸信号交易。

本节里为了保持示例简单，只画原始 K 线。如果要进一步增强，可以把事件筛出来后用 monitor 的 scatter/line 叠加到 K 线图上。


In [24]:
smc_kline_view = smc_features.select("datetime", "open", "high", "low", "close").tail(350)
smc_kline_plot = col("datetime", "open", "high", "low", "close").monitor(
    "smc_price_kline",
    show_axis_label=True,
).kline().runtime()
smc_kline_plot.plot(smc_kline_view, open_in_jupyter=True, auto_open=False, height=620)


## 5. 常见组合方式

实际策略里一般不会直接“看到 SMC 事件就交易”，而是把它作为结构过滤或状态特征。例如：

- `BOS > 0`：结构向上突破，可以作为只做多或加强多头信号的过滤；
- `BOS < 0`：结构向下突破，可以作为只做空或加强空头信号的过滤；
- `CHOCH != 0`：结构性质可能发生变化，适合用来降低原趋势信号权重；
- `FVG != 0`：出现不平衡区间，后续可以观察是否回补；
- `Liquidity != 0`：附近有多个相近高低点，可以观察是否被扫；
- `BrokenHigh/Low`：是否突破上一周期区间，适合和趋势/波动率过滤结合；
- `CurrentRetracement%`：把回撤幅度数值化，便于做区间筛选或参数优化。

qust 写法上，这些都只是列，可以继续：

```python
smc_expr.filter(col("BOS__structure") > col.lit(0))
smc_expr.with_cols((col("FVG__fvg") != col.lit(0)).alias("has_fvg"))
smc_expr.over("ticker", "ct")
```

如果你要做多合约研究，不要把所有合约混在一个结构里识别；应该让 SMC 表达式在 `ticker + ct` 维度独立执行：

```python
col("open", "high", "low", "close").smc.bos_choch(12).over("ticker", "ct")
```

最后再强调一次：SMC 指标更适合作为结构描述和过滤条件。是否能形成交易策略，还需要回测、样本外验证、手续费/滑点、风险控制和仓位管理。


In [5]:
smc_signal_sample = (
    smc_features
    .filter(
        (pl.col("BOS__structure") != 0)
        | (pl.col("CHOCH__structure") != 0)
        | (pl.col("FVG__fvg") != 0)
        | (pl.col("Liquidity__liq") != 0)
    )
    .select(
        "datetime",
        "close",
        "FVG__fvg",
        "Top__fvg",
        "Bottom__fvg",
        "BOS__structure",
        "CHOCH__structure",
        "Level__structure",
        "Liquidity__liq",
        "Level__liq",
    )
)

print("signal rows:", smc_signal_sample.height)
smc_signal_sample.head(20)


signal rows: 0


datetime,close,FVG__fvg,Top__fvg,Bottom__fvg,BOS__structure,CHOCH__structure,Level__structure,Liquidity__liq,Level__liq
datetime[ms],f64,i32,f64,f64,i32,i32,f64,i32,f64
